# VFD Crystallisation Operator — Demonstration

This notebook demonstrates the crystallisation operator:
1. Multi-mode initial state
2. Evolution under gradient flow $\dot{\Psi} = -\nabla F[\Psi]$
3. Convergence to stable attractor
4. Comparison vs random collapse
5. Spectral reweighting

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from vfd.crystallisation.operator import (
    closure_residual,
    energy_functional,
    coherence_metric,
    crystallisation_functional,
    crystallise,
    crystallisation_flow,
    spectral_reweight,
    crystallisation_timescale,
    hessian_check,
    CrystallisationParams,
)

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Multi-Mode Initial State

Create a random superposition of modes with varying phases and amplitudes.

In [ ]:
n = 6  # number of modes
psi_init = np.random.randn(n) + 1j * np.random.randn(n)
psi_init = psi_init / np.linalg.norm(psi_init)

print('Initial state amplitudes:', np.abs(psi_init).round(4))
print('Initial state phases (rad):', np.angle(psi_init).round(4))
print(f'Initial R = {closure_residual(psi_init):.4f}')
print(f'Initial E = {energy_functional(psi_init):.4f}')
print(f'Initial Q = {coherence_metric(psi_init):.4f}')
print(f'Initial F = {crystallisation_functional(psi_init):.4f}')
print(f'Crystallisation timescale tau = {crystallisation_timescale(psi_init):.4f}')

## 2. Crystallisation via Gradient Flow

In [ ]:
params = CrystallisationParams(
    alpha=1.0, beta=0.5, gamma=0.8,
    eta=0.003, max_steps=5000, tol=1e-8,
    record_trajectory=True,
)

result = crystallise(psi_init, params)

print(f'Converged: {result.converged}')
print(f'Steps: {result.steps}')
print(f'F initial: {result.F_trajectory[0]:.6f}')
print(f'F final:   {result.F_final:.6f}')
print(f'Final amplitudes: {np.abs(result.psi_star).round(4)}')
print(f'Final phases: {np.angle(result.psi_star).round(4)}')
print(f'Final Q = {coherence_metric(result.psi_star):.4f}')

In [ ]:
# Plot F(t) — monotonic descent
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(result.F_trajectory, 'b-', linewidth=1.5)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('F[Ψ]')
axes[0].set_title('Crystallisation Functional F(t) — Monotonic Descent')
axes[0].grid(True, alpha=0.3)

# Plot mode amplitudes over time
psi_traj = np.array(result.psi_trajectory)
for i in range(n):
    axes[1].plot(np.abs(psi_traj[:, i]), label=f'Mode {i}')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('|c_i|')
axes[1].set_title('Mode Amplitude Evolution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Stability Verification

Check that the crystallised state is a stable attractor (positive definite Hessian).

In [ ]:
is_stable, eigenvalues = hessian_check(result.psi_star, params)
print(f'Stable attractor: {is_stable}')
print(f'Hessian eigenvalues (sorted): {np.sort(eigenvalues).round(6)}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(eigenvalues)), np.sort(eigenvalues))
ax.axhline(y=0, color='r', linestyle='--', alpha=0.5)
ax.set_xlabel('Eigenvalue index')
ax.set_ylabel('λ')
ax.set_title('Hessian Eigenvalues at Ψ★')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Comparison: Crystallisation vs Random Collapse

Run 50 trials. Crystallisation always converges to the same attractor from the same initial state. Random collapse gives different results each time.

In [ ]:
n_trials = 50

# Crystallisation: deterministic from same init
crystal_results = []
for _ in range(n_trials):
    r = crystallise(psi_init, params)
    crystal_results.append(r.F_final)

# Random collapse: pick eigenstate with |c_i|^2 probability
collapse_F = []
basis = np.eye(n, dtype=np.complex128)
probs = np.abs(psi_init)**2
probs = probs / probs.sum()
for _ in range(n_trials):
    k = np.random.choice(n, p=probs)
    phi_k = basis[k]
    collapse_F.append(crystallisation_functional(phi_k, params))

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(crystal_results, bins=20, alpha=0.7, label='Crystallisation F★', color='blue')
ax.hist(collapse_F, bins=20, alpha=0.7, label='Random Collapse F', color='red')
ax.set_xlabel('F[Ψ]')
ax.set_ylabel('Count')
ax.set_title('Crystallisation (deterministic) vs Random Collapse (stochastic)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Crystallisation F: mean={np.mean(crystal_results):.6f}, std={np.std(crystal_results):.2e}')
print(f'Random collapse F: mean={np.mean(collapse_F):.6f}, std={np.std(collapse_F):.4f}')

## 5. Spectral Reweighting

Demonstrate mode selection: modes with low residual and high coherence dominate.

In [ ]:
# Per-mode properties (synthetic)
c = np.array([0.2, 0.25, 0.15, 0.2, 0.1, 0.1])
R_modes = np.array([0.9, 0.1, 0.7, 0.3, 0.8, 0.6])  # Mode 1 has lowest R
E_modes = np.array([0.3, 0.2, 0.5, 0.4, 0.6, 0.3])
Q_modes = np.array([0.2, 0.9, 0.3, 0.7, 0.1, 0.4])  # Mode 1 has highest Q

# Sweep reweighting strength
strengths = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(n)
width = 0.12

for idx, s in enumerate(strengths):
    c_new = spectral_reweight(c, R_modes, E_modes, Q_modes, mu=s, nu=s*0.5, rho=s)
    ax.bar(x + idx * width, np.abs(c_new), width, label=f'μ=ρ={s}', alpha=0.8)

ax.set_xlabel('Mode index')
ax.set_ylabel('|c̃_i|')
ax.set_title('Spectral Reweighting: Mode Selection Under Increasing Constraint Strength')
ax.set_xticks(x + width * len(strengths) / 2)
ax.set_xticklabels([f'Mode {i}' for i in range(n)])
ax.legend(title='Strength')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Mode 1 (lowest R, highest Q) dominates as strength increases.')

## 6. Continuous Flow Visualisation

Integrate $d\Psi/dt = -\nabla F[\Psi]$ and visualise component-level evolution.

In [ ]:
psi_flow_init = np.random.randn(4) + 1j * np.random.randn(4)
psi_flow_init = psi_flow_init / np.linalg.norm(psi_flow_init)

flow_params = CrystallisationParams(alpha=1.0, beta=0.5, gamma=0.8)
psi_final, F_t, psi_t = crystallisation_flow(
    psi_flow_init, dt=0.002, num_steps=1500, params=flow_params
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# F(t)
axes[0, 0].plot(F_t, 'b-', linewidth=1.5)
axes[0, 0].set_title('F(t) under gradient flow')
axes[0, 0].set_xlabel('Time step')
axes[0, 0].set_ylabel('F')
axes[0, 0].grid(True, alpha=0.3)

# Amplitudes
for i in range(4):
    axes[0, 1].plot(np.abs(psi_t[:, i]), label=f'|ψ_{i}|')
axes[0, 1].set_title('Component amplitudes')
axes[0, 1].set_xlabel('Time step')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Phases
for i in range(4):
    axes[1, 0].plot(np.angle(psi_t[:, i]), label=f'arg(ψ_{i})')
axes[1, 0].set_title('Component phases')
axes[1, 0].set_xlabel('Time step')
axes[1, 0].set_ylabel('Phase (rad)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# R, E, Q over time
R_t = [closure_residual(psi_t[t]) for t in range(len(F_t))]
E_t_vals = [energy_functional(psi_t[t]) for t in range(len(F_t))]
Q_t = [coherence_metric(psi_t[t]) for t in range(len(F_t))]

axes[1, 1].plot(R_t, label='R (residual)', color='red')
axes[1, 1].plot(E_t_vals, label='E (energy)', color='orange')
axes[1, 1].plot(Q_t, label='Q (coherence)', color='green')
axes[1, 1].set_title('Functional components over time')
axes[1, 1].set_xlabel('Time step')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Final F = {F_t[-1]:.6f}')
print(f'Final R = {R_t[-1]:.6f}, E = {E_t_vals[-1]:.6f}, Q = {Q_t[-1]:.6f}')

## 7. Multi-Initial-Condition Basin Analysis

Launch crystallisation from 20 random initial states and observe attractor convergence.

In [ ]:
n_inits = 20
final_states = []
final_F_vals = []

basin_params = CrystallisationParams(
    alpha=1.0, beta=0.5, gamma=0.8,
    eta=0.003, max_steps=5000, tol=1e-8,
)

for seed in range(n_inits):
    rng = np.random.default_rng(seed)
    psi0 = rng.normal(size=4) + 1j * rng.normal(size=4)
    psi0 = psi0 / np.linalg.norm(psi0)
    r = crystallise(psi0, basin_params)
    final_states.append(r.psi_star)
    final_F_vals.append(r.F_final)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(n_inits), final_F_vals, color='steelblue', alpha=0.8)
ax.set_xlabel('Initial condition index')
ax.set_ylabel('F★')
ax.set_title('Final F values from 20 different initial conditions')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'F★ range: [{min(final_F_vals):.4f}, {max(final_F_vals):.4f}]')
print(f'Number of distinct basins (approx): {len(set(round(f, 2) for f in final_F_vals))}')

## Summary

This demonstration shows:
1. **Monotonic descent**: F decreases strictly under gradient flow
2. **Attractor convergence**: the state crystallises into a stable configuration
3. **Determinism**: same initial conditions always yield same result (unlike random collapse)
4. **Mode selection**: spectral reweighting amplifies coherent, low-residual modes
5. **Stability**: Hessian analysis confirms crystallised states are genuine minima